# 12 — Dynamic Validation (Synthetic Expert Trajectories)

**Question:** Can we recover parameter trajectories across sessions?
How do static per-session methods compare to dynamic SBI?

**Protocol:**
1. Generate synthetic expert animals with one parameter drifting
   via random walk (known ground truth)
2. Static GS: pooled + per-session
3. Static SBI: pooled + per-session
4. Dynamic SBI: joint multi-session fit with temporal link functions
5. Compare: trajectory recovery accuracy

In [ ]:
%matplotlib inline
from shared_setup import *

import time
from analysis.grid_search import compute_grid_search_fit, DEFAULT_GRID, COARSE_GRID
from behav_utils.analysis.update_matrix import compute_update_matrix
from behav_utils.plotting.update_matrix import plot_um
from models.BE_core import BEParams, BEModel
from models.SC_core import SCParams, SCModel
from scripts.config import GS_BURN_IN, GS_N_BINS, BASE_SEED

SBI_OK = False
try:
    import torch
    from inference.amortised import AmortisedSBI, compute_pooled_stats
    from inference.fitting import SBIFitter
    from inference.types import ConstantSpec, RandomWalkSpec
    from behav_utils.data.structures import FittingData
    SBI_OK = True
    print(f'SBI available (torch {torch.__version__})')
except ImportError as e:
    print(f'SBI not available: {e}')

apply_style()

## 1. Configuration

In [ ]:
QUICK = True

if QUICK:
    N_PER_MODEL = 1
    N_SESSIONS = 10
    TRIALS_PER_SESSION = 300
    GS_GRID = COARSE_GRID
    SBI_SIMS_STATIC = 2_000
    SBI_SIMS_DYNAMIC = 2_000
    N_TRAJECTORY_SAMPLES = 200
    BURN_IN = 500
    LINK_TYPES = ['randomwalk']
else:
    N_PER_MODEL = 5
    N_SESSIONS = 15
    TRIALS_PER_SESSION = 350
    GS_GRID = DEFAULT_GRID
    SBI_SIMS_STATIC = 50_000
    SBI_SIMS_DYNAMIC = 30_000
    N_TRAJECTORY_SAMPLES = 500
    BURN_IN = GS_BURN_IN
    LINK_TYPES = ['constant', 'randomwalk', 'gp']


SEED = BASE_SEED
FIT_TARGET = 'update_matrix'
SIGMA_DRIFT = 0.2

# Colours for dynamic SBI link types
LINK_COLOURS = {
    'constant':   '#9E9E9E',   # grey
    'randomwalk': PALETTE[2],
    'gp':         PALETTE[3] if len(PALETTE) > 3 else '#8E24AA',
}
LINK_STYLES = {
    'constant':   '--',
    'randomwalk': '-',
    'gp':         '-.',
}

print(f'Mode: {"QUICK" if QUICK else "FULL"}')
print(f'Animals: {N_PER_MODEL} BE + {N_PER_MODEL} SC')
print(f'Sessions: {N_SESSIONS} x {TRIALS_PER_SESSION} trials')
print(f'Ground truth drift: sigma={SIGMA_DRIFT}')

## 2. Generate synthetic dynamic animals

In [ ]:
def generate_dynamic_animal(
    model_type, base_params, varying_param, sigma_drift,
    n_sessions, trials_per_session, burn_in, animal_id, seed,
):
    rng = np.random.default_rng(seed)
    if model_type == 'BE':
        Model, Params = BEModel, BEParams
        bounds = BEParams.get_bounds()
    else:
        Model, Params = SCModel, SCParams
        bounds = SCParams.get_bounds()

    lo, hi = bounds[varying_param]
    base_val = getattr(base_params, varying_param)
    trajectory = [base_val]
    for _ in range(n_sessions - 1):
        new_val = np.clip(trajectory[-1] + rng.normal(0, sigma_drift), lo, hi)
        trajectory.append(new_val)
    trajectory = np.array(trajectory)

    state = Model.create_initial_state(burn_in=burn_in, params=base_params, seed=seed)
    sessions = []
    for s_idx in range(n_sessions):
        p_dict = {k: getattr(base_params, k) for k in Params.get_param_names()}
        p_dict[varying_param] = float(trajectory[s_idx])
        params = Params(**p_dict)
        stimuli = rng.uniform(-1, 1, trials_per_session)
        categories = (stimuli > 0).astype(int)
        choices, _, state, _ = Model.simulate_session(
            params, state, stimuli, categories, rng, return_history=False)
        sessions.append({'stimuli': stimuli, 'choices': choices, 'categories': categories})

    return {'animal_id': animal_id, 'true_model': model_type,
            'true_params_base': base_params, 'varying_param': varying_param,
            'true_trajectory': trajectory, 'sessions': sessions}

animals = []
for i in range(N_PER_MODEL):
    base = BEParams(sigma_percep=0.15, A_repulsion=0.08, eta_learning=0.30, eta_relax=0.12)
    animals.append(generate_dynamic_animal(
        'BE', base, 'eta_learning', SIGMA_DRIFT,
        N_SESSIONS, TRIALS_PER_SESSION, BURN_IN, f'BE_dyn_{i:02d}', SEED + i))
for i in range(N_PER_MODEL):
    base = SCParams(sigma_percep=0.15, A_repulsion=0.08, gamma=0.50, sigma_update=0.25)
    animals.append(generate_dynamic_animal(
        'SC', base, 'gamma', SIGMA_DRIFT,
        N_SESSIONS, TRIALS_PER_SESSION, BURN_IN, f'SC_dyn_{i:02d}', SEED + 100 + i))

print(f'{len(animals)} dynamic synthetic animals')
for a in animals:
    t = a['true_trajectory']
    print(f"  {a['animal_id']}: {a['true_model']}, {a['varying_param']} = {t[0]:.3f} -> {t[-1]:.3f} (range {t.max()-t.min():.3f})")

## 3. Visualise ground truth

In [ ]:
# 3a. True parameter trajectories
n_a = len(animals)
fig, axes = plt.subplots(1, n_a, figsize=(5 * n_a, 3.5), squeeze=False)
for col, a in enumerate(animals):
    ax = axes[0, col]
    ax.plot(a['true_trajectory'], 'k-o', lw=2, ms=4)
    ax.set_xlabel('Session')
    ax.set_ylabel(a['varying_param'])
    ax.set_title(f"{a['animal_id']} ({a['true_model']})")
fig.suptitle('Ground Truth Trajectories', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3b. Per-session UMs (first animal only)
a0 = animals[0]
n_show = min(5, N_SESSIONS)
fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 3))
if n_show == 1: axes = [axes]
for s_idx in range(n_show):
    sess = a0['sessions'][s_idx]
    um, _, _ = compute_update_matrix(
        sess['stimuli'], sess['choices'], sess['categories'],
        n_bins=GS_N_BINS, trial_filter='post_correct')
    plot_um(um, ax=axes[s_idx],
            title=f"S{s_idx} ({a0['varying_param']}={a0['true_trajectory'][s_idx]:.3f})")
fig.suptitle(f"{a0['animal_id']}: Per-Session UMs", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Static GS

In [ ]:
print('=== Static GS ===')
t0 = time.time()
for a in animals:
    mt = a['true_model']
    grid = GS_GRID[mt]
    vp = a['varying_param']

    # 4a. Pooled
    p_stim = np.concatenate([s['stimuli'] for s in a['sessions']])
    p_ch = np.concatenate([s['choices'] for s in a['sessions']])
    p_cat = np.concatenate([s['categories'] for s in a['sessions']])
    gs_pooled = compute_grid_search_fit(p_stim, p_ch, p_cat, model_type=mt,
                                grid=grid, burn_in=BURN_IN, fit_target=FIT_TARGET, seed=SEED)
    a['gs_pooled'] = gs_pooled['best_params']

    # 4b. Per-session
    gs_per = []
    for sess in a['sessions']:
        r = compute_grid_search_fit(sess['stimuli'], sess['choices'], sess['categories'],
                            model_type=mt, grid=grid, burn_in=BURN_IN,
                            fit_target=FIT_TARGET, seed=SEED)
        gs_per.append(r['best_params'])
    a['gs_per_session'] = gs_per

    print(f"  {a['animal_id']}: pooled {vp}={a['gs_pooled'][vp]:.3f}, "
          f"per-session [{min(p[vp] for p in gs_per):.3f}, {max(p[vp] for p in gs_per):.3f}]")
print(f'  Done in {time.time() - t0:.1f}s')

## 5. Static SBI

In [ ]:
if SBI_OK:
    print('=== Training 1-session AmortisedSBI ===')
    curriculum_1 = [('uniform', 1)]
    trainers_1 = {}
    for mt in ['be', 'sc']:
        trainer = AmortisedSBI(model_type=mt, curriculum=curriculum_1,
                               trials_per_session=TRIALS_PER_SESSION, burn_in=BURN_IN)
        trainer.train(n_simulations=SBI_SIMS_STATIC, seed=SEED)
        trainers_1[mt] = trainer

    print('=== Conditioning ===')
    for a in animals:
        mt = a['true_model'].lower()
        vp = a['varying_param']
        # 5a. Pooled
        p_stim = np.concatenate([s['stimuli'] for s in a['sessions']])
        p_ch = np.concatenate([s['choices'] for s in a['sessions']])
        p_cat = np.concatenate([s['categories'] for s in a['sessions']])
        cond_p = trainers_1[mt].condition_from_arrays(p_stim, p_ch, p_cat, n_samples=300)
        a['sbi_pooled'] = cond_p['point_estimate']
        # 5b. Per-session
        sbi_per = []
        for sess in a['sessions']:
            cond = trainers_1[mt].condition_from_arrays(
                sess['stimuli'], sess['choices'], sess['categories'], n_samples=300)
            sbi_per.append(cond['point_estimate'])
        a['sbi_per_session'] = sbi_per
        print(f"  {a['animal_id']}: pooled {vp}={a['sbi_pooled'][vp]:.3f}, "
              f"per-session [{min(p[vp] for p in sbi_per):.3f}, {max(p[vp] for p in sbi_per):.3f}]")
else:
    print('SBI not available')

## 6. Dynamic SBI

Joint multi-session fit using `SBIFitter` with `RandomWalkSpec`
on the varying parameter. Other parameters held constant.

In [ ]:
if SBI_OK:
    from inference.types import ConstantSpec, RandomWalkSpec, GPSpec

    LINK_BUILDERS = {
        'constant': lambda b: ConstantSpec(bounds=b),
        'randomwalk': lambda b: RandomWalkSpec(bounds=b, sigma_drift=SIGMA_DRIFT * 2),
        'gp': lambda b: GPSpec(bounds=b),
    }

    print('=== Dynamic SBI ===')
    for a in animals:
        mt = a['true_model'].lower()
        vp = a['varying_param']
        aid = a['animal_id']

        fd = FittingData.from_arrays(a['sessions'], animal_id=aid)

        if mt == 'be':
            bounds = BEParams.get_bounds()
            all_params = BEParams.get_param_names()
        else:
            bounds = SCParams.get_bounds()
            all_params = SCParams.get_param_names()

        a['dynamic_trajectories'] = {}

        for lt in LINK_TYPES:
            param_links = {}
            for pn in all_params:
                if pn == vp:
                    param_links[pn] = LINK_BUILDERS[lt](bounds[pn])
                else:
                    param_links[pn] = ConstantSpec(bounds=bounds[pn])

            t0 = time.time()
            fitter = SBIFitter(fitting_data=fd, model_type=mt,
                               param_links=param_links, burn_in=BURN_IN)
            result = fitter.train(n_simulations=SBI_SIMS_DYNAMIC, seed=SEED)
            trajectories = fitter.extract_trajectories(
                result, n_samples=N_TRAJECTORY_SAMPLES)
            a['dynamic_trajectories'][lt] = trajectories
            print(f'  {aid}/{lt}: {time.time() - t0:.1f}s')
else:
    print('SBI not available')

## 7. Trajectory comparison plots

In [ ]:
n_a = len(animals)
fig, axes = plt.subplots(1, n_a, figsize=(6 * n_a, 4.5), squeeze=False)

for col, a in enumerate(animals):
    ax = axes[0, col]
    vp = a['varying_param']
    sx = np.arange(N_SESSIONS)

    # True trajectory
    ax.plot(sx, a['true_trajectory'], 'k-', lw=2.5, label='True', zorder=5)

    # GS pooled (horizontal line)
    ax.axhline(a['gs_pooled'][vp], color=PALETTE[0], ls='--',
               lw=1.2, alpha=0.6, label='GS pooled')

    # GS per-session
    gs_vals = [p[vp] for p in a['gs_per_session']]
    ax.scatter(sx, gs_vals, c=PALETTE[0], s=25, alpha=0.5,
               zorder=3, label='GS per-sess')

    # SBI pooled + per-session
    if 'sbi_pooled' in a:
        ax.axhline(a['sbi_pooled'][vp], color=PALETTE[1], ls='--',
                   lw=1.2, alpha=0.6, label='SBI pooled')
        sbi_vals = [p[vp] for p in a['sbi_per_session']]
        ax.scatter(sx, sbi_vals, c=PALETTE[1], s=25, marker='D',
                   alpha=0.5, zorder=3, label='SBI per-sess')

    # Dynamic SBI trajectories (one per link type)
    if 'dynamic_trajectories' in a:
        for lt in LINK_TYPES:
            if lt not in a['dynamic_trajectories']:
                continue
            traj = a['dynamic_trajectories'][lt]
            if vp not in traj:
                continue

            t_data = traj[vp]
            median = np.array(t_data['median'])
            lo_ci = np.array(t_data.get('ci_low', t_data.get('q05', median)))
            hi_ci = np.array(t_data.get('ci_high', t_data.get('q95', median)))

            lc = LINK_COLOURS.get(lt, PALETTE[2])
            ls = LINK_STYLES.get(lt, '-')

            ax.plot(sx[:len(median)], median, ls,
                    color=lc, lw=2, zorder=4, label=f'Dynamic ({lt})')
            ax.fill_between(sx[:len(median)], lo_ci, hi_ci,
                            color=lc, alpha=0.12, zorder=2)

    ax.set_xlabel('Session')
    ax.set_ylabel(vp)
    ax.set_title(f"{a['animal_id']} ({a['true_model']})")
    ax.legend(fontsize=7, loc='best')

fig.suptitle('Trajectory Recovery: All Methods', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Recovery metrics

In [ ]:
def trajectory_rmse(true, recovered):
    v = np.isfinite(recovered)
    return float(np.sqrt(np.mean((true[v] - recovered[v]) ** 2))) if v.sum() > 0 else np.nan

def trajectory_corr(true, recovered):
    v = np.isfinite(recovered)
    if v.sum() < 3:
        return np.nan
    from scipy.stats import pearsonr
    r, _ = pearsonr(true[v], recovered[v])
    return float(r)

rows = []
for a in animals:
    vp = a['varying_param']
    true = a['true_trajectory']
    aid = a['animal_id']

    # GS pooled
    gs_p = np.full(N_SESSIONS, a['gs_pooled'][vp])
    rows.append({'animal': aid, 'method': 'GS pooled',
                 'rmse': trajectory_rmse(true, gs_p),
                 'r': trajectory_corr(true, gs_p)})

    # GS per-session
    gs_vals = np.array([p[vp] for p in a['gs_per_session']])
    rows.append({'animal': aid, 'method': 'GS per-session',
                 'rmse': trajectory_rmse(true, gs_vals),
                 'r': trajectory_corr(true, gs_vals)})

    # SBI pooled
    if 'sbi_pooled' in a:
        sbi_p = np.full(N_SESSIONS, a['sbi_pooled'][vp])
        rows.append({'animal': aid, 'method': 'SBI pooled',
                     'rmse': trajectory_rmse(true, sbi_p),
                     'r': trajectory_corr(true, sbi_p)})

        sbi_vals = np.array([p[vp] for p in a['sbi_per_session']])
        rows.append({'animal': aid, 'method': 'SBI per-session',
                     'rmse': trajectory_rmse(true, sbi_vals),
                     'r': trajectory_corr(true, sbi_vals)})

    # Dynamic SBI — one row per link type
    if 'dynamic_trajectories' in a:
        for lt in LINK_TYPES:
            if lt not in a['dynamic_trajectories']:
                continue
            traj = a['dynamic_trajectories'][lt]
            if vp not in traj:
                continue
            dyn_vals = np.array(traj[vp]['median'])
            rows.append({'animal': aid, 'method': f'Dynamic ({lt})',
                         'rmse': trajectory_rmse(true, dyn_vals),
                         'r': trajectory_corr(true, dyn_vals)})

metrics_df = pd.DataFrame(rows)

# Summary
summary = (metrics_df.groupby('method')[['rmse', 'r']]
           .agg(['mean', 'std']).round(4))
print('=== Recovery Metrics (mean +/- std across animals) ===')
print(summary)
print()

# Per-animal
print('=== Per-Animal Detail ===')
print(metrics_df.to_string(index=False))

## 9. Summary

**Expected ranking** (best to worst RMSE):
Dynamic SBI (RW) > SBI per-session > GS per-session > pooled methods

**Key takeaways:**
- Pooled methods give a single number — useless for trajectories
- Per-session methods track direction but are noisy
- Dynamic SBI with RandomWalk borrows strength across sessions,
  producing the smoothest and most accurate trajectory

**Limitation:** RandomWalk assumes smooth drift — cannot represent
abrupt strategy switches. That motivates the SLDS + per-state
model selection pipeline (NB 30-31).

**Next:** NB 13 tests the full trajectory scenario
(learning -> expert -> shift).